In [23]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14

In [24]:
# Ensure the project root is on the import path so `src` can be imported.
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break
else:
    raise RuntimeError('Run this notebook from inside the StockForecastRisk repository.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.data.loader import PROCESSED_FEATURES_PATH, load_processed_features
raw_prices = load_processed_features()

print(f'Loaded {len(raw_prices):,} rows for {raw_prices["symbol"].nunique():,} tickers')
print(f'Source: {PROCESSED_FEATURES_PATH}')
raw_prices.head()

Loaded 1,943,662 rows for 497 tickers
Source: D:\StockForecastRisk\data\processed\sp500_features.parquet


,date,symbol,security,gics_sector,gics_sub_industry,adj_close,close,high,low,open,...,treasury_2y,fed_funds_rate,vix_term_ratio,yield_curve_slope,vix_change_5d,hy_credit_spread_change_5d,yield_curve_slope_change_5d,vix_change_20d,hy_credit_spread_change_20d,yield_curve_slope_change_20d
0,2010-01-04,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,19.772951,22.389128,22.625179,22.267525,22.453505,...,1.09,0.12,0.880105,2.76,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,19.558167,22.145924,22.331903,22.002861,22.324751,...,1.01,0.12,0.864225,2.76,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,19.488672,22.067240,22.174536,22.002861,22.067240,...,1.01,0.12,0.878899,2.84,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,19.463404,22.038628,22.045780,21.816881,22.017166,...,1.03,0.10,0.882407,2.82,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-08,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,19.457090,22.031473,22.067240,21.745352,21.917025,...,0.96,0.11,0.863333,2.87,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
raw_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 1943662 entries, 0 to 1943661
Data columns (total 84 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   date                          datetime64[ms]
 1   symbol                        str           
 2   security                      str           
 3   gics_sector                   str           
 4   gics_sub_industry             str           
 5   adj_close                     float64       
 6   close                         float64       
 7   high                          float64       
 8   low                           float64       
 9   open                          float64       
 10  volume                        float64       
 11  history_rows                  int64         
 12  short_history                 bool          
 13  sma_10                        float64       
 14  sma_20                        float64       
 15  sma_50                        float64      

In [26]:
print("dtype:", raw_prices["date"].dtype)
print("distinct dates:", raw_prices["date"].nunique())          # expect ~4169, NOT 1.9M
print("sample values:", raw_prices["date"].head(3).tolist())
print("any time component?:")
print(pd.to_datetime(raw_prices["date"]).dt.time.value_counts().head())  # expect all 00:00:00

dtype: datetime64[ms]
distinct dates: 4169
sample values: [Timestamp('2010-01-04 00:00:00'), Timestamp('2010-01-05 00:00:00'), Timestamp('2010-01-06 00:00:00')]
any time component?:
date
00:00:00    1943662
Name: count, dtype: int64


In [27]:
# pick a pre-2021 date that EXISTS in the good macro file
mask = raw_prices["date"] == pd.Timestamp("2015-06-15")
print("rows on 2015-06-15:", mask.sum())
print(raw_prices.loc[mask, ["date", "symbol", "vix", "treasury_10y", "fed_funds_rate"]].head())
print("vix null on this date?:", raw_prices.loc[mask, "vix"].isna().all())

rows on 2015-06-15: 455
            date symbol    vix  treasury_10y  fed_funds_rate
1370  2015-06-15      A  15.39          2.36            0.13
5539  2015-06-15   AAPL  15.39          2.36            0.13
8954  2015-06-15   ABBV  15.39          2.36            0.13
14538 2015-06-15    ABT  15.39          2.36            0.13
18707 2015-06-15   ACGL  15.39          2.36            0.13
vix null on this date?: False


In [28]:
raw_prices.describe(include='all')

,date,symbol,security,gics_sector,gics_sub_industry,adj_close,close,high,low,open,...,treasury_2y,fed_funds_rate,vix_term_ratio,yield_curve_slope,vix_change_5d,hy_credit_spread_change_5d,yield_curve_slope_change_5d,vix_change_20d,hy_credit_spread_change_20d,yield_curve_slope_change_20d
count,1943662,1943662,1943662,1943662,1943662,1.943614e+06,1.943614e+06,1.943614e+06,1.943614e+06,1.943614e+06,...,1.943165e+06,1.943165e+06,1.943165e+06,1.943165e+06,1.941081e+06,369034.000000,1.941081e+06,1.937328e+06,364134.000000,1.937328e+06
unique,NaN,497,497,11,127,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,A,Agilent Technologies,Industrials,Health Care Equipment,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4169,4169,301602,63443,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2018-07-08 20:45:42.328000,NaN,NaN,NaN,NaN,1.019857e+02,1.111685e+02,1.124318e+02,1.098515e+02,1.111532e+02,...,1.754356e+00,1.552244e+00,8.940898e-01,9.306236e-01,1.218099e-01,-0.006236,-1.785253e-03,7.018983e-02,-0.021102,-7.495298e-03
min,2010-01-04 00:00:00,NaN,NaN,NaN,NaN,2.032893e-01,2.220000e-01,2.262500e-01,2.162500e-01,2.180000e-01,...,9.000000e-02,4.000000e-02,7.103584e-01,-1.080000e+00,-1.874000e+01,-0.520000,-3.800000e-01,-3.067000e+01,-0.840000,-6.500000e-01
25%,2014-06-13 00:00:00,NaN,NaN,NaN,NaN,2.820960e+01,3.521284e+01,3.561000e+01,3.480676e+01,3.521000e+01,...,3.800000e-01,1.000000e-01,8.430556e-01,2.500000e-01,-1.190000e+00,-0.080000,-4.000000e-02,-2.310000e+00,-0.160000,-8.000000e-02
50%,2018-08-17 00:00:00,NaN,NaN,NaN,NaN,5.499384e+01,6.574000e+01,6.645000e+01,6.501000e+01,6.573000e+01,...,9.700000e-01,3.800000e-01,8.840304e-01,8.200000e-01,-2.000000e-02,-0.020000,-4.440892e-16,-1.800000e-01,-0.040000,-1.000000e-02
75%,2022-08-23 00:00:00,NaN,NaN,NaN,NaN,1.102706e+02,1.225900e+02,1.239100e+02,1.212000e+02,1.225700e+02,...,3.360000e+00,2.410000e+00,9.349376e-01,1.580000e+00,1.190000e+00,0.050000,3.000000e-02,1.910000e+00,0.090000,6.000000e-02
max,2026-07-31 00:00:00,NaN,NaN,NaN,NaN,9.924400e+03,9.924400e+03,9.964770e+03,9.794000e+03,9.914170e+03,...,5.190000e+00,5.330000e+00,1.343719e+00,2.910000e+00,3.353000e+01,1.190000,5.900000e-01,5.839000e+01,1.380000,6.200000e-01


In [29]:
summary = {
        "total_rows": len(raw_prices),
        "total_columns": raw_prices.shape[1],
        "number_of_symbols": raw_prices["symbol"].nunique(),
        "number_of_securities": raw_prices["security"].nunique(),
        "number_of_gics_sectors": raw_prices["gics_sector"].nunique(),
        "number_of_gics_sub_industries": raw_prices["gics_sub_industry"].nunique(),
        "start_date": raw_prices["date"].min(),
        "end_date": raw_prices["date"].max(),
        "total_trading_days": raw_prices["date"].nunique(),
    }
    
display(pd.DataFrame(summary.items(), columns=["metric", "value"]))

,metric,value
0,total_rows,1943662
1,total_columns,84
2,number_of_symbols,497
3,number_of_securities,497
4,number_of_gics_sectors,11
5,number_of_gics_sub_industries,127
6,start_date,2010-01-04 00:00:00
7,end_date,2026-07-31 00:00:00
8,total_trading_days,4169


In [30]:
summary = pd.DataFrame({
        "column": raw_prices.columns,
        "dtype": raw_prices.dtypes.astype(str).values,
        "non_null_count": raw_prices.notna().sum().values,
        "null_count": raw_prices.isna().sum().values,
        "null_pct": (raw_prices.isna().mean() * 100).values,
    })

display(summary.sort_values("null_pct", ascending=False))

,column,dtype,non_null_count,null_count,null_pct
82,hy_credit_spread_change_20d,float64,364134,1579528,81.265570
79,hy_credit_spread_change_5d,float64,369034,1574628,81.013468
72,ig_credit_spread,float64,370994,1572668,80.912628
71,hy_credit_spread,float64,370994,1572668,80.912628
16,sma_200,float64,1841129,102533,5.275248
...,...,...,...,...,...
61,ticker_id,int64,1943662,0,0.000000
58,day_of_week,int32,1943662,0,0.000000
60,quarter,int32,1943662,0,0.000000
62,sector_id,int64,1943662,0,0.000000


In [31]:
rows_per_symbol = raw_prices.groupby("symbol").size()

rows_per_symbol_summary = pd.DataFrame({
    "count_symbols": [rows_per_symbol.count()],
    "min_rows": [rows_per_symbol.min()],
    "q1_rows": [rows_per_symbol.quantile(0.25)],
    "median_rows": [rows_per_symbol.median()],
    "mean_rows": [rows_per_symbol.mean()],
    "q3_rows": [rows_per_symbol.quantile(0.75)],
    "max_rows": [rows_per_symbol.max()],
    "std_rows": [rows_per_symbol.std()],
})

display(rows_per_symbol_summary)

,count_symbols,min_rows,q1_rows,median_rows,mean_rows,q3_rows,max_rows,std_rows
0,497,33,4169.0,4169.0,3910.788732,4169.0,4169,743.858924


In [32]:
date_coverage_by_symbol = (
    raw_prices
    .groupby("symbol")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        trading_days=("date", "nunique"),
        rows=("date", "count")
    )
    .reset_index()
    .sort_values("trading_days", ascending=False)
)

display(date_coverage_by_symbol.head(20))
display(date_coverage_by_symbol.tail(20))

,symbol,first_date,last_date,trading_days,rows
479,WFC,2010-01-04,2026-07-31,4169,4169
478,WELL,2010-01-04,2026-07-31,4169,4169
477,WEC,2010-01-04,2026-07-31,4169,4169
476,WDC,2010-01-04,2026-07-31,4169,4169
474,WBD,2010-01-04,2026-07-31,4169,4169
473,WAT,2010-01-04,2026-07-31,4169,4169
472,WAB,2010-01-04,2026-07-31,4169,4169
471,VZ,2010-01-04,2026-07-31,4169,4169
470,VTRS,2010-01-04,2026-07-31,4169,4169
469,VTR,2010-01-04,2026-07-31,4169,4169


,symbol,first_date,last_date,trading_days,rows
126,DDOG,2019-09-19,2026-07-31,1725,1725
348,OTIS,2020-03-19,2026-07-31,1600,1600
72,CARR,2020-03-19,2026-07-31,1600,1600
364,PLTR,2020-09-30,2026-07-31,1465,1465
124,DASH,2020-12-09,2026-07-31,1416,1416
3,ABNB,2020-12-10,2026-07-31,1415,1415
171,EXE,2021-02-10,2026-07-31,1374,1374
102,COIN,2021-04-14,2026-07-31,1331,1331
37,APP,2021-04-15,2026-07-31,1330,1330
225,HOOD,2021-07-29,2026-07-31,1257,1257


In [33]:
missing_stats = pd.DataFrame({
        "column": raw_prices.columns,
        "missing_count": raw_prices.isna().sum().values,
        "missing_pct": (raw_prices.isna().mean() * 100).values,
        "non_missing_count": raw_prices.notna().sum().values,
    })

missing_stats = missing_stats.sort_values(
        "missing_count",
        ascending=False
    )

display(missing_stats[missing_stats["missing_count"] > 0])

,column,missing_count,missing_pct,non_missing_count
82,hy_credit_spread_change_20d,1579528,81.265570,364134
79,hy_credit_spread_change_5d,1574628,81.013468,369034
72,ig_credit_spread,1572668,80.912628,370994
71,hy_credit_spread,1572668,80.912628,370994
16,sma_200,102533,5.275248,1841129
...,...,...,...,...
6,close,48,0.002470,1943614
9,open,48,0.002470,1943614
8,low,48,0.002470,1943614
7,high,48,0.002470,1943614


In [34]:
columns = min(raw_prices.select_dtypes(include="number").columns.tolist(), key=len)

columns = raw_prices.select_dtypes(include="number").columns.to_list()
feature_stats = pd.DataFrame(index=columns)

feature_stats["count"] = raw_prices[columns].count()
feature_stats["missing_count"] = raw_prices[columns].isna().sum()
feature_stats["missing_pct"] = raw_prices[columns].isna().mean() * 100

feature_stats["mean"] = raw_prices[columns].mean()
feature_stats["median"] = raw_prices[columns].median()
feature_stats["std"] = raw_prices[columns].std()

feature_stats["min"] = raw_prices[columns].min()
feature_stats["q1"] = raw_prices[columns].quantile(0.25)
feature_stats["q3"] = raw_prices[columns].quantile(0.75)
feature_stats["max"] = raw_prices[columns].max()

feature_stats["skew"] = raw_prices[columns].skew()
feature_stats["kurtosis"] = raw_prices[columns].kurtosis()

display(feature_stats.reset_index(names="feature"))

,feature,count,missing_count,missing_pct,mean,median,std,min,q1,q3,max,skew,kurtosis
0,adj_close,1943614,48,0.002470,101.985736,54.993841,234.194743,0.203289,28.209604,110.27058,9924.400391,18.645509,510.349639
1,close,1943614,48,0.002470,111.168468,65.739998,234.728824,0.222,35.212842,122.589996,9924.400391,18.439875,502.920836
2,high,1943614,48,0.002470,112.431805,66.449997,237.580489,0.22625,35.610001,123.910004,9964.769531,18.432191,502.28719
3,low,1943614,48,0.002470,109.851519,65.010002,231.892934,0.21625,34.80676,121.199997,9794.0,18.474579,504.934729
4,open,1943614,48,0.002470,111.153183,65.730003,234.724166,0.218,35.209999,122.57,9914.169922,18.44953,503.521666
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,hy_credit_spread_change_5d,369034,1574628,81.013468,-0.006236,-0.02,0.138368,-0.52,-0.08,0.05,1.19,1.720071,12.752183
74,yield_curve_slope_change_5d,1941081,2581,0.132791,-0.001785,-0.0,0.067465,-0.38,-0.04,0.03,0.59,0.4599,4.594603
75,vix_change_20d,1937328,6334,0.325880,0.07019,-0.18,5.221497,-30.67,-2.31,1.91,58.39,1.792698,16.827975
76,hy_credit_spread_change_20d,364134,1579528,81.265570,-0.021102,-0.04,0.264401,-0.84,-0.16,0.09,1.38,1.021306,4.536023
